## Chapter 1 Exercises

Use the following workbook to practice what you've learned. There are 3 exercises here. Please feel free to extend the existing scenarios, experiment and test other samples you find while reading the Agent Framework documentation.

### Client Setup

Make sure you have `client` available from the Section 1 notebook. If not, run the cell below.

In [ ]:
import os
from dotenv import load_dotenv
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

load_dotenv(override=True)

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)
print("Client ready.")

### Exercise 1 — Implement an Agent with Multiple Function Tools

In Section 1 you saw how to define tools as standalone functions and give them to an `Agent`. In practice, tools that share state (a database connection, an in-memory data store) are best grouped into a **class**.

In this exercise you will build a small **ClaimsTracker** — an internal tool to log and query insurance claims during a call.

#### Task:
1. Create a `ClaimsTracker` class with the following methods (each will become a tool):
    - `log_claim(policy_number, description, severity)` — Registers a new claim against a policy. Severity is one of: critical, high, medium, low.
    - `list_claims()` — Returns all logged claims.
    - `filter_by_severity(severity)` — Returns claims that match the given severity level.

2. Annotate parameters with `Annotated` + `Field(description=...)` so the model understands each argument.
3. Decorate each method with `@tool(approval_mode="never_require")`.
4. Create an `Agent` with clear instructions and register the class methods as tools.
5. Test the agent with a multi-turn conversation using a session.

This exercise reinforces: class-based tools with shared state, `@tool` decorator, `Annotated` parameters, multi-turn sessions.

### Implement ClaimsTracker

Fill in the TODOs below. Each method should be decorated with `@tool(approval_mode="never_require")` and have a docstring (the model reads it).

In [ ]:
from typing import Annotated, List, Dict
from pydantic import Field


class ClaimsTracker:
    def __init__(self):
        # TODO: Initialize an empty list to store claims.
        # Each claim is a dict: {"policy_number": str, "description": str, "severity": str}
        pass

    @tool(approval_mode="never_require")
    def log_claim(
        self,
        policy_number: Annotated[str, Field(description="Athora Netherlands policy number, e.g. NL-2031-887.")],
        description: Annotated[str, Field(description="Short description of what happened.")],
        severity: Annotated[str, Field(description="Severity level: critical, high, medium, or low.")],
    ) -> str:
        # TODO: Append the claim to self.claims and return a confirmation message.
        pass

    @tool(approval_mode="never_require")
    def list_claims(self) -> str:
        # TODO: Return all claims. If none exist, return "No claims logged yet."
        pass

    @tool(approval_mode="never_require")
    def filter_by_severity(
        self,
        severity: Annotated[str, Field(description="Severity to filter by: critical, high, medium, or low.")],
    ) -> str:
        # TODO: Return claims matching the given severity. If none match, say so.
        pass


# TODO: Create an instance of ClaimsTracker


# TODO: Create an Agent with instructions and tools from the tracker instance
# Hint: tools=[tracker.log_claim, tracker.list_claims, tracker.filter_by_severity]


# TODO: Create a session for multi-turn conversation
# Hint: session = agent.create_session()


# Test the agent
user_messages = [
    "A customer on policy NL-2031-887 reports water damage to their property. High severity.",
    "Log another claim: policy NL-4408-552, the policyholder passed away last week. Critical severity.",
    "Also log a low severity claim on NL-2031-887: customer wants to update their address.",
    "Show me all claims.",
    "Which claims are critical?",
]

for msg in user_messages:
    print(f"*** Rep: {msg}")
    response = await agent.run(msg, session=session)
    print(f"*** PolicyPal: {response.text}\n")

<details>
    <summary>See the solution</summary>

```python
from typing import Annotated, List, Dict
from pydantic import Field


class ClaimsTracker:
    def __init__(self):
        self.claims: List[Dict[str, str]] = []

    @tool(approval_mode="never_require")
    def log_claim(
        self,
        policy_number: Annotated[str, Field(description="Athora Netherlands policy number, e.g. NL-2031-887.")],
        description: Annotated[str, Field(description="Short description of what happened.")],
        severity: Annotated[str, Field(description="Severity level: critical, high, medium, or low.")],
    ) -> str:
        """Log a new insurance claim against a policy."""
        self.claims.append({
            "policy_number": policy_number,
            "description": description,
            "severity": severity,
        })
        return f"Claim logged for {policy_number} (severity: {severity}): {description}"

    @tool(approval_mode="never_require")
    def list_claims(self) -> str:
        """List all logged claims."""
        if not self.claims:
            return "No claims logged yet."
        lines = []
        for i, c in enumerate(self.claims, 1):
            lines.append(f"{i}. [{c['severity'].upper()}] {c['policy_number']}: {c['description']}")
        return "\n".join(lines)

    @tool(approval_mode="never_require")
    def filter_by_severity(
        self,
        severity: Annotated[str, Field(description="Severity to filter by: critical, high, medium, or low.")],
    ) -> str:
        """Return claims matching the given severity level."""
        filtered = [c for c in self.claims if c["severity"].lower() == severity.lower()]
        if not filtered:
            return f"No claims with severity '{severity}'."
        lines = []
        for i, c in enumerate(filtered, 1):
            lines.append(f"{i}. {c['policy_number']}: {c['description']}")
        return "\n".join(lines)


# Create instance
tracker = ClaimsTracker()

# Create agent
agent = Agent(
    client=client,
    name="PolicyPal",
    instructions=(
        "You are PolicyPal, an internal assistant for Athora Netherlands reps. "
        "Use the claims tools to log and query insurance claims. "
        "Be concise and confirm each action clearly."
    ),
    tools=[tracker.log_claim, tracker.list_claims, tracker.filter_by_severity],
)

# Create session for multi-turn
session = agent.create_session()

user_messages = [
    "A customer on policy NL-2031-887 reports water damage to their property. High severity.",
    "Log another claim: policy NL-4408-552, the policyholder passed away last week. Critical severity.",
    "Also log a low severity claim on NL-2031-887: customer wants to update their address.",
    "Show me all claims.",
    "Which claims are critical?",
]

for msg in user_messages:
    print(f"*** Rep: {msg}")
    response = await agent.run(msg, session=session)
    print(f"*** PolicyPal: {response.text}\n")
```
</details>

### Exercise 2 - Build a Memory Extraction Provider

In Section 1 you saw `CustomerProfileMemory` which uses `before_run` to inject pre-loaded facts. In this exercise you will build a provider that uses both hooks:

- `after_run` — **observe** what the user said and extract preferences automatically.
- `before_run` — **inject** those stored preferences into the agent's context on the next turn.

### Scenario

You are building an HR assistant that helps employees choose benefits plans. Employees often state preferences during the conversation ("I prefer plans with lower deductibles", "I like remote-friendly perks"). The provider should learn these and personalize future recommendations — without the user repeating themselves.

### The Task

1. Create a `SimpleMemoryProvider` context provider that:
   - In `after_run`, scans the user's message for preference keywords ("I prefer", "I like", "always", "please use") and stores matching statements.
   - In `before_run`, injects all stored preferences as extra instructions so the agent respects them on future turns.

2. Create an `Agent` with this provider and test it with a multi-turn session.

3. Verify that turn 3 reflects preferences stated in turns 1 and 2

### Implement SimpleMemoryProvider

Fill in the TODOs. Refer to the `CustomerProfileMemory` in Section 1 for the signatures.

In [ ]:
PREFERENCE_KEYWORDS = # TODO: Define a list of keywords to look for in user messages


class SimpleMemoryProvider(ContextProvider):
    """Learns user preferences from conversation and injects them on future turns."""

    DEFAULT_SOURCE_ID = "user_memories"

    def __init__(self) -> None:
        super().__init__(self.DEFAULT_SOURCE_ID)
        self._memories: list[str] = []

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        # TODO: If there are no stored memories, return early.
        # TODO: Format memories as bullet points and inject them

        pass

    async def after_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        # TODO: Loop over context.input_messages
        # TODO: For each message, get the text
        # TODO: Check if the text contains any preference keywords
        # TODO: If yes, append the text to memory
        pass


# TODO: Create an instance of SimpleMemoryProvider


# TODO: Create an Agent with context_providers=[...] and instructions


# TODO: Create a session for multi-turn conversation


# Test conversation
user_messages = [
    "I have children so pediatric coverage is key.",
    "I want to find plans that cover mental health care with no copay.",
    "Please use cost comparison given in monthly terms, not yearly.",
]

for msg in user_messages:
    print(f"*** User: {msg}")
    response = await agent.run(msg, session=session)
    print(f"*** Agent: {response.text}\n")

# Verify what was captured
print(f"--- Stored memories: {memory._memories}")


<details>
<summary>See the solution</summary>

```python
PREFERENCE_KEYWORDS = ["i prefer", "i like", "always", "please use", "never", "is key", "i want"]


class SimpleMemoryProvider(ContextProvider):
    """Learns user preferences from conversation and injects them on future turns."""

    DEFAULT_SOURCE_ID = "user_memories"

    def __init__(self) -> None:
        super().__init__(self.DEFAULT_SOURCE_ID)
        self._memories: list[str] = []

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        if not self._memories:
            return
        bullets = "\n".join(f"- {m}" for m in self._memories)
        context.extend_instructions(
            self.source_id,
            f"Remembered facts about the user. Always respect these:\n{bullets}",
        )

    async def after_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        for msg in context.input_messages:
            text = getattr(msg, "text", "")
            if not isinstance(text, str):
                continue
            if any(kw in text.lower() for kw in PREFERENCE_KEYWORDS):
                self._memories.append(text)


# Create provider and agent
memory = SimpleMemoryProvider()

agent = Agent(
    client=client,
    name="HRAssistant",
    instructions=(
        "You are a helpful HR assistant. Use any remembered facts to personalize responses."
    ),
    context_providers=[memory],
)

session = agent.create_session()

user_messages = [
    "I have children so pediatric coverage is key.",
    "I want to find plans that cover mental health care with no copay.",
    "Please use cost comparison given in monthly terms, not yearly.",
]

for msg in user_messages:
    print(f"*** User: {msg}")
    response = await agent.run(msg, session=session)
    print(f"*** Agent: {response.text}\n")

print(f"--- Stored memories: {memory._memories}")
```

</details>

This exercise uses simple keyword matching to detect preferences. In a real application, you'd implement more robust natural language processing. Feel free to extend this example with more complex context provider logic. Refer to the official Agent Framework samples for inspiration and additional patterns.

## Exercise 3 - Managing Meeting Rooms with an Agent

This example shows how an agent can perform actions on meeting room states using a custom tool class. We define a `RoomTools` class that manages a list of rooms, each represented by a `RoomModel` typed dictionary.

The agent can:
- List all rooms and their current availability.
- Get the details of a specific room by ID.
- Book a room (mark it as unavailable and set the organiser).
- Change room settings (e.g. projector on/off, whiteboard request).

By exposing these methods as tools, the agent can interact with and modify the environment dynamically.

### Step 1 - Define RoomModel and RoomTools

In [ ]:
from typing import TypedDict, Annotated, List, Optional


class RoomModel(TypedDict):
    """Typed dictionary for meeting room properties."""
    id: int
    name: str
    capacity: int
    available: bool
    has_projector: bool
    has_whiteboard: bool
    booked_by: str | None


class RoomTools:
    """Tool class for managing meeting rooms."""

    def __init__(self, rooms: list[RoomModel]):
        self.rooms = rooms

    def get_rooms(self) -> List[RoomModel]:
        """Get a list of all meeting rooms and their current state."""
        return self.rooms

    def get_room(
        self, id: Annotated[int, "The ID of the room"]
    ) -> Optional[RoomModel]:
        """Get details of a specific meeting room."""
        for room in self.rooms:
            if room["id"] == id:
                return room
        return None

    def book_room(
        self,
        id: Annotated[int, "The ID of the room to book"],
        organiser: Annotated[str, "Name of the person booking the room"],
    ) -> dict:
        """Book a meeting room. Sets it as unavailable and records the organiser."""
        for room in self.rooms:
            if room["id"] == id:
                if not room["available"]:
                    return {"error": f"Room '{room['name']}' is already booked by {room['booked_by']}"}
                previous = room.copy()
                room["available"] = False
                room["booked_by"] = organiser
                return {"previous": previous, "current": room}
        return {"error": "Room not found"}

    def update_room(
        self,
        id: Annotated[int, "The ID of the room"],
        has_projector: Annotated[Optional[bool], "Set projector availability"] = None,
        has_whiteboard: Annotated[Optional[bool], "Set whiteboard availability"] = None,
    ) -> dict:
        """Update room equipment settings (projector, whiteboard)."""
        for room in self.rooms:
            if room["id"] == id:
                previous = room.copy()
                if has_projector is not None:
                    room["has_projector"] = has_projector
                if has_whiteboard is not None:
                    room["has_whiteboard"] = has_whiteboard
                return {"previous": previous, "current": room}
        return {"error": "Room not found"}

Now we provide initial room data and create an instance. 
This instance holds the current state of all rooms and exposes methods the agent can use as tools.

In [ ]:
rooms = [
    {"id": 1, "name": "Amsterdam", "capacity": 8,  "available": True,  "has_projector": True,  "has_whiteboard": True,  "booked_by": None},
    {"id": 2, "name": "Rotterdam", "capacity": 4,  "available": False, "has_projector": False, "has_whiteboard": True,  "booked_by": "Jan"},
    {"id": 3, "name": "Utrecht",   "capacity": 12, "available": True,  "has_projector": True,  "has_whiteboard": False, "booked_by": None},
]

room_tools = RoomTools(rooms=rooms)

Now let's create the agent, attach the tools, and run a command to manage the rooms.

In [ ]:
agent = Agent(
    client=client,
    name="RoomManager",
    instructions="""
    You manage meeting room bookings and equipment.
    - Always check room availability before booking.
    - Do NOT change equipment settings unless explicitly asked.
    - When booking, always confirm which room was booked and for whom.""",
    tools=[room_tools.get_rooms, room_tools.get_room, room_tools.book_room, room_tools.update_room],
)

result = await agent.run(
    "Book the Amsterdam room for Lisa and make sure the projector is enabled"
)

print(result)
print("\nFinal room data:", room_tools.rooms)

### Step 2 - Add Middleware to an Agent

In the following exercise, we'll add middleware for easier debugging and understanding of agent behavior, because currently we can't see anything besides the agent's final response.

We'll add:
- **`@function_middleware`** -- intercepts each tool call, logs function name, arguments, and result
- **`@chat_middleware`** -- intercepts each LLM request, logs message count and response
- **`@agent_middleware`** -- wraps the entire agent run, logs total duration

The decorator-based style uses `call_next()` (no arguments) to invoke the next step in the pipeline. 
After `call_next()` returns, you can inspect `context.result` for the outcome.

**Key context attributes:**

| Middleware | Context type | Useful attributes |
|---|---|---|
| `@agent_middleware` | `AgentContext` | `context.agent`, `context.messages`, `context.result` |
| `@chat_middleware` | `ChatContext` | `context.messages`, `context.result` |
| `@function_middleware` | `FunctionInvocationContext` | `context.function.name`, `context.arguments`, `context.result` |

In [ ]:
import logging
import time
from agent_framework import agent_middleware, chat_middleware, function_middleware

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", force=True
)
logger = logging.getLogger(__name__)


# TODO: Implement agent logging middleware
# - Use the appropriate decorator and signature
    start = time.perf_counter()
# - Log before execution
# - Call await call_next() to run the agent
    duration = time.perf_counter() - start


# TODO: Implement chat_logging_middleware
# - Use the appropriate decorator and signature
# - Log before sending messages to the LLM
# - Call await call_next()
# - Log after receiving the response


# TODO: Implement function_logging_middleware
# - Use the appropriate decorator and signature
# - Signature: async def function_logging_middleware(context, call_next)
# - Log the function name and arguments BEFORE calling call_next()
    start = time.perf_counter()
    await call_next()
    duration = time.perf_counter() - start
# - Log the function name, duration, and result AFTER


# Reset rooms for a fresh run
rooms = [
    {"id": 1, "name": "Amsterdam", "capacity": 8,  "available": True,  "has_projector": True,  "has_whiteboard": True,  "booked_by": None},
    {"id": 2, "name": "Rotterdam", "capacity": 4,  "available": False, "has_projector": False, "has_whiteboard": True,  "booked_by": "Jan"},
    {"id": 3, "name": "Utrecht",   "capacity": 12, "available": True,  "has_projector": True,  "has_whiteboard": False, "booked_by": None},
]
room_tools = RoomTools(rooms=rooms)

agent = Agent(
    client=client,
    name="RoomManager",
    instructions="""
    You manage meeting room bookings and equipment.
    - Always check room availability before booking.
    - Do NOT change equipment settings unless explicitly asked.
    - When booking, always confirm which room was booked and for whom.""",
    tools=[room_tools.get_rooms, room_tools.get_room, room_tools.book_room, room_tools.update_room],
    # TODO: Add middleware= parameter with a list of your middleware functions
)

result = await agent.run(
    "Book the Utrecht room for Sophie and enable the whiteboard"
)

print("Assistant >", result)

<details>
    <summary>See the solution</summary>

```python
import logging
import time
from agent_framework import agent_middleware, chat_middleware, function_middleware

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", force=True
)
logger = logging.getLogger(__name__)


@agent_middleware
async def agent_logging_middleware(context, call_next):
    start = time.perf_counter()
    logger.info("[Agent] Starting execution")

    await call_next()

    duration = time.perf_counter() - start
    logger.info(f"[Agent] Completed in {duration:.4f}s")


@chat_middleware
async def chat_logging_middleware(context, call_next):
    logger.info(f"[Chat] Sending {len(context.messages)} messages to LLM")

    await call_next()

    logger.info("[Chat] Response received")


@function_middleware
async def function_logging_middleware(context, call_next):
    logger.info(f"[Function] Calling {context.function.name} | Args: {context.arguments}")

    start = time.perf_counter()
    await call_next()
    duration = time.perf_counter() - start

    logger.info(f"[Function] {context.function.name} completed in {duration:.6f}s | Result: {context.result}")


rooms = [
    {"id": 1, "name": "Amsterdam", "capacity": 8,  "available": True,  "has_projector": True,  "has_whiteboard": True,  "booked_by": None},
    {"id": 2, "name": "Rotterdam", "capacity": 4,  "available": False, "has_projector": False, "has_whiteboard": True,  "booked_by": "Jan"},
    {"id": 3, "name": "Utrecht",   "capacity": 12, "available": True,  "has_projector": True,  "has_whiteboard": False, "booked_by": None},
]
room_tools = RoomTools(rooms=rooms)

agent = Agent(
    client=client,
    name="RoomManager",
    instructions="""
    You manage meeting room bookings and equipment.
    - Always check room availability before booking.
    - Do NOT change equipment settings unless explicitly asked.
    - When booking, always confirm which room was booked and for whom.""",
    tools=[room_tools.get_rooms, room_tools.get_room, room_tools.book_room, room_tools.update_room],
    middleware=[agent_logging_middleware, chat_logging_middleware, function_logging_middleware],
)

result = await agent.run(
    "Book the Utrecht room for Sophie and enable the whiteboard"
)

print("Assistant >", result)
```
</details>


## Chapter 1 Complete! 🎉

  You've completed Section 1 of the workshop. Here's what you've learned:

   - Agents - how to create an agent with instructions, tools, and a chat client
   - Function tools - exposing Python methods as callable tools for the LLM
   - Context providers - injecting dynamic context into the prompt with before_run and after_run hooks
   - Middleware - intercepting agent execution at every layer (agent, chat, function) for logging and 
  control

These are the foundational building blocks for everything that follows. Feel free to experiment, play around with other code samples and break things!

In the next sections, we'll use these concepts to build multi-agent workflows, add observability with OpenTelemetry, and evaluate agent quality. See you in Chapter 2!
